# 소행성 관측 계획 수립하기

* 이 노트북을 구글 코랩에서 실행하고자 한다면 [파일] - [드라이브에 사본 저장]을 하여 본인의 소유로 만든 후에 코드를 실행하거나 수정할 수 있습니다.

* 이 파일은 실제 수업에 사용하므로 필요에 따라 예고 없이 변경될 수 있습니다.

* If you have any questions or comments on this document, please email me(Kiehyun.Park@gmail.com).

* 이 파일(문서)는 공교육 현장에서 수업시간에 자유롭게 사용할 수 있으나, 다른 목적으로 사용할 시에는 사전에 연락을 주셔서 상의해 주시기 바랍니다.

이 자료는 What's observable로 부터 특정한 날 관측할 수 있는 소행성의 목록을 만드는 과정을 설명합니다.

## 필요한 환경

이 프로젝트를 위해서는 아래의 모듈이 필요합니다.

> numpy, pandas, matplotlib, astroquery, requests, astropy, astropy.units, certifi, version_information


### 구글 코랩에 한글 폰트 설치

matplotlib에서 한글을 사용하기 위해서는 한글 폰트가 필요하다. 구글 코랩에서 현재의 Jupyter notebook을 실행한다면 아래 코드를 실행 한 후 런타임 다시 시작을 해 줘야 한글을 사용할 수 있을 것이다.

In [12]:
# !sudo apt-get install -y fonts-nanum
# !sudo fc-cache -fv
# !rm ~/.cache/matplotlib -rf

### 런타임 다시 시작

위의 셀을 실행한 다음 반드시 다음 과정을 잊지 말자.

* [메뉴]-[런타임]-[런터임 다시 시작]

* [메뉴]-[런타임]-[이전 셀 실행]을 해주어야 한다.

### 한글 폰트 사용

위에서 한글 폰트를 설치하고, 런타임 다시시작을 했다면 구글 코랩에서 폰트 경로를 설정하여 한글 사용이 가능해 진다.

In [13]:
#visualization
import matplotlib as mpl
import matplotlib.pylab as plb
import matplotlib.pyplot as plt

# 브라우저에서 바로 그려지도록
%matplotlib inline

# 그래프에 retina display 적용
%config InlineBackend.figure_format = 'retina'

# Colab 의 한글 폰트 설정
plt.rc('font', family='NanumBarunGothic')

# 유니코드에서  음수 부호설정
mpl.rc('axes', unicode_minus=False)

### 모듈 설치 및 버전 확인

아래 셀을 실행하면 이 노트북을 실행하는데 필요한 모듈을 설치하고 파이썬 및 관련 모듈의 버전을 확인할 수 있다.

In [14]:
import importlib, sys, subprocess
packages = "numpy, pandas, matplotlib, astroquery, requests, astropy, astropy.units, certifi, version_information" # required modules
pkgs = packages.split(", ")
for pkg in pkgs :
    if not importlib.util.find_spec(pkg):
        subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])
        print(f"**** {pkg} module is now installed.")
    else:
        print(f"******** {pkg} module is already installed.")
%load_ext version_information
import time
now = time.strftime("%Y-%m-%d %H:%M:%S (%Z = GMT%z)")
print(f"This notebook was generated at {now} ")

vv = %version_information {packages}
for i, pkg in enumerate(vv.packages):
    print(f"{i} {pkg[0]:10s} {pkg[1]:s}")

******** numpy module is already installed.
******** pandas module is already installed.
******** matplotlib module is already installed.
******** astroquery module is already installed.
******** requests module is already installed.
******** astropy module is already installed.
******** astropy.units module is already installed.
******** certifi module is already installed.
******** version_information module is already installed.
The version_information extension is already loaded. To reload it, use:
  %reload_ext version_information
This notebook was generated at 2023-12-03 19:40:55 (KST = GMT+0900) 
0 Python     3.11.5 64bit [GCC 11.2.0]
1 IPython    8.15.0
2 OS         Linux 5.15.0 89 generic x86_64 with glibc2.31
3 numpy      1.26.0
4 pandas     2.1.1
5 matplotlib 3.8.0
6 astroquery 0.4.6
7 requests   2.31.0
8 astropy    5.3.4
9 astropy.units The 'astropy.units' distribution was not found and is required by the application
10 certifi    2023.07.22
11 version_information 1.0.4


## 날짜와 시간 다루기

### datetime.datetime

Python 내장 모듈인 datetime을 사용하면 날짜와 시각 객체를 다룰 수 있습니다.

#### 표준시와 세계시

In [15]:
from datetime import datetime, timedelta, timezone

dt_kst_now = datetime.now()
print("현재 시각(KST=GMT+9) :", dt_kst_now)

timezone_utc = timezone(timedelta(hours=-9))
dt_utc_now = dt_kst_now.astimezone(timezone_utc)
print("현재 시각(UTC) :", dt_utc_now)

현재 시각(KST=GMT+9) : 2023-12-03 19:40:55.563316
현재 시각(UTC) : 2023-12-03 01:40:55.563316-09:00


#### fits header의 시간 형식

fits header의 시간 형식은 '2023-09-11T20:19:18'의 형태를 띠고 있는데 datetime.strptime을 이용하면 날짜시간 형식을 바로 datetime 객체로 만들 수 있습니다.

이를 세계시로 바꾸려면 timedelta를 이용할 수 있습니다.

In [16]:
start_dt_kst_str = '2023-09-11T20:19:18'
end_dt_kst_str = '2023-12-31T00:00:00'

start_dt_kst = datetime.strptime(start_dt_kst_str, '%Y-%m-%dT%H:%M:%S')
print(start_dt_kst)
print(type(start_dt_kst))

start_dt_utc = start_dt_kst + timedelta(hours=-9)
print(start_dt_utc)
print(type(start_dt_utc))

2023-09-11 20:19:18
<class 'datetime.datetime'>
2023-09-11 11:19:18
<class 'datetime.datetime'>


#### string 형태로 변환하기

datetime 객체를 string으로 바꿀때는 strftime 메쏘들ㄹ 이용하면 됩니다.

In [17]:

start_dt_utc_str = start_dt_utc.strftime('%Y-%m-%dT%H:%M:%S')
print("start_dt_utc_str :", start_dt_utc_str)
print("type(start_dt_utc_str) :", type(start_dt_utc_str))


start_dt_utc_str : 2023-09-11T11:19:18
type(start_dt_utc_str) : <class 'str'>


#### datetime 이용하여 list 생성

아래와 같이 하면, 시작 시각과 끝나는 시각, 그리고 시간 간격을 정해 주고, Python의 datatime 객체를 이용하여 일정 시간 간격의 목록(list)을 만을 수 있다.

In [18]:
from datetime import datetime, timedelta
start_time_text = "2017-01-01"
end_time_text = "2017-06-01"
dt = 1
def datetime_range(start_dt, end_dt, delta):
    '''
        Parameters
        ----------
        start_dt : datetime.datetime
        end_dt : datetime.datetime
        delta : int
    '''
    current = start_dt
    while current < end_dt:
        yield current
        current += delta

dts = [dt.strftime('%Y-%m-%d') for dt in
            datetime_range(datetime.fromisoformat(start_time_text), datetime.fromisoformat(end_time_text),
            timedelta(days=dt))]

print("len(dts):", len(dts))
print("dts", dts)

len(dts): 151
dts ['2017-01-01', '2017-01-02', '2017-01-03', '2017-01-04', '2017-01-05', '2017-01-06', '2017-01-07', '2017-01-08', '2017-01-09', '2017-01-10', '2017-01-11', '2017-01-12', '2017-01-13', '2017-01-14', '2017-01-15', '2017-01-16', '2017-01-17', '2017-01-18', '2017-01-19', '2017-01-20', '2017-01-21', '2017-01-22', '2017-01-23', '2017-01-24', '2017-01-25', '2017-01-26', '2017-01-27', '2017-01-28', '2017-01-29', '2017-01-30', '2017-01-31', '2017-02-01', '2017-02-02', '2017-02-03', '2017-02-04', '2017-02-05', '2017-02-06', '2017-02-07', '2017-02-08', '2017-02-09', '2017-02-10', '2017-02-11', '2017-02-12', '2017-02-13', '2017-02-14', '2017-02-15', '2017-02-16', '2017-02-17', '2017-02-18', '2017-02-19', '2017-02-20', '2017-02-21', '2017-02-22', '2017-02-23', '2017-02-24', '2017-02-25', '2017-02-26', '2017-02-27', '2017-02-28', '2017-03-01', '2017-03-02', '2017-03-03', '2017-03-04', '2017-03-05', '2017-03-06', '2017-03-07', '2017-03-08', '2017-03-09', '2017-03-10', '2017-03-11', '

### astropy.time

표준시나 세계시 정도는 datetime을 사용해도 무방하지만 천문학에서 사용하는 시간에는 조금 부족합니다. 천문학에서 사용하는 시간계는 datetime 모듈보다는 astropy.time 모듈을 사용하기를 권장합니다.

#### 세계시

In [19]:
from astropy.time import Time

t_kst_now = Time.now()
print("현재 시각(KST=GMT+9) :", t_kst_now)

t_ut_now = Time(datetime.utcnow(), scale='utc', location=('127d', '37d'))
print("현재 시각(UTC) :", t_ut_now)

print("type(t_ut_now) ;", type(t_ut_now))

현재 시각(KST=GMT+9) : 2023-12-03 10:40:55.594685
현재 시각(UTC) : 2023-12-03 10:40:55.595217
type(t_ut_now) ; <class 'astropy.time.core.Time'>


#### dir함수로 메쏘드 알아보기


In [20]:
print("dir(t_ut_now) ;", dir(t_ut_now))

dir(t_ut_now) ; ['FORMATS', 'SCALES', 'T', '_APPLICABLE_FUNCTIONS', '_METHOD_FUNCTIONS', '__abstractmethods__', '__add__', '__array_function__', '__array_priority__', '__bool__', '__class__', '__copy__', '__deepcopy__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattr__', '__getattribute__', '__getitem__', '__getnewargs__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__iter__', '__le__', '__len__', '__lt__', '__module__', '__ne__', '__new__', '__radd__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__setitem__', '__sizeof__', '__str__', '__sub__', '__subclasshook__', '__weakref__', '_abc_impl', '_advanced_index', '_apply', '_astropy_column_attrs', '_call_erfa', '_format', '_get_delta_tdb_tt', '_get_delta_ut1_utc', '_get_time_fmt', '_init_from_vals', '_make_value_equivalent', '_match_shape', '_set_delta_tdb_tt', '_set_delta_ut1_utc', '_set_scale', '_shaped_like_input', '_sid_time_or_earth_rot_ang', '_

#### Julian day 구하기

In [21]:
print("t_ut_now.to_value(format='jd') ;", t_ut_now.to_value(format='jd'))

t_ut_now.to_value(format='jd') ; 2460281.945087908


#### 항성시

In [22]:
print("t_ut_now.sidereal_time('mean') :", t_ut_now.sidereal_time('mean'))

t_ut_now.sidereal_time('mean') : 23h56m57.41770781s


#### astropy.time 이용하여 list 생성

일정 시간 간격으로 시뮬레이션 하려면 시간 간격의 리스트를 미리 만들어 두면 편리할 것입니다. 다음은 Python의 astropy.time을 이용하여 일정 시간 간격의 목록(list)을 만드는 방법입니다.

In [23]:
import numpy as np
from astropy.time import Time, TimeDelta

start_time_text = "2023-12-03"
end_time_text = "2023-12-31"
dt = 1

def astropyTime_range(start_t, end_t, delta):
    '''
        Parameters
        ----------
        start_t : datetime.datetime
        end_t : datetime.datetime
        delta : int
    '''
    current = start_t
    while current < end_t:
        yield current
        current += delta

dts = [dt for dt in
        astropyTime_range(Time(start_time_text), Time(end_time_text),
        TimeDelta(dt, format='jd'))]

print(len(dts))
print(dts)

28
[<Time object: scale='utc' format='iso' value=2023-12-03 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-04 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-05 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-06 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-07 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-08 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-09 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-10 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-11 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-12 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-13 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-14 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-15 00:00:00.000>, <Time object: scale='utc' format='iso' value=2023-12-16 00:00:00.000>, <T

## What's observable

[What's observable](https://ssd.jpl.nasa.gov/tools/sbwobs.html) 에서는 관측소 코드와 관측 조건들을 설정하여 소행성과 혜성의 관측 대상의 정보를 얻을 수 있습니다.


### Horizons API

[Horizons API](https://ssd-api.jpl.nasa.gov/doc/horizons.html)를 제공하고 있어 requests로 받아아올 url을 만들수 있습니다.



#### 검색 조건 설정

먼저 조건들을 변수로 설정해 보겠습니다.

In [24]:
#############################################
# variables
mpc_code='P64' # Observer Location: GSHS Observatory, Suwon [code: P64]

#관측 시각 설정 (현재 시각)
t_ut_now = Time(datetime.utcnow(), scale='utc', location=('127d', '37d'))
print("type(obs_dt_utc) :", type(t_ut_now))
print("현재 시각(UTC) :", t_ut_now)

#관측 시각 설정 (임의의 시각)
# t_ut_now = Time(datetime(2023, 10, 2, 10, 0, 0), scale='utc', location=('127d', '37d'))
# print("type(obs_dt_utc) :", type(t_ut_now))
# print("현재 시각(UTC) :", t_ut_now)

obs_datetime = t_ut_now.strftime('%Y-%m-%d %H:%M:%S')
print("obs_datetime :", obs_datetime)
obs_date = t_ut_now.strftime('%Y%m%d')
print("obs_date :", obs_date)

elev_min=60  #
time_min=15
vmag_min=6
vmag_max=13
list_num=100
sort='trans'

type(obs_dt_utc) : <class 'astropy.time.core.Time'>
현재 시각(UTC) : 2023-12-03 10:41:15.233755
obs_datetime : 2023-12-03 10:41:15
obs_date : 20231203


#### url 만들기

In [25]:
# Define API URL and SPK filename:
url = 'https://ssd-api.jpl.nasa.gov/sbwobs.api'
spk_filename = 'spk_file.bsp'

# Get the requested SPK-ID from the command-line:
if (len(sys.argv)) == 1:
  print("please specify SPK-ID on the command-line")
  sys.exit(2)
spkid = sys.argv[1]

# Build the appropriate URL for this API request:
# IMPORTANT: You must encode the "=" as "%3D" and the ";" as "%3B" in the
#            Horizons COMMAND parameter specification.
url += f"?sb-kind=a&mpc-code={mpc_code}&obs-time={str(obs_datetime)}&elev-min={elev_min}&time-min={time_min}"
url += f"&vmag-max={vmag_max}&vmag-min={vmag_min}&optical=true&fmt-ra-dec=true&mag-required=true&output-sort={sort}&maxoutput={list_num}"

print("url :", url)

url : https://ssd-api.jpl.nasa.gov/sbwobs.api?sb-kind=a&mpc-code=P64&obs-time=2023-12-03 10:41:15&elev-min=60&time-min=15&vmag-max=13&vmag-min=6&optical=true&fmt-ra-dec=true&mag-required=true&output-sort=trans&maxoutput=100


### reauests.get

api에서 제공하는 내용을 json 형태로 받아 보겠습니다.

In [28]:
import json
import requests

# Submit the API request and decode the JSON-response:
response = requests.get(url)
try:
  data = json.loads(response.text)
  print(data)
except ValueError:
  print("Unable to decode JSON results")

{'signature': {'source': 'NASA/JPL Small-Body Observability API', 'version': '1.0'}, 'obs_constraints': {'time-min': '15', 'mpc-code': 'P64', 'fmt-ra-dec': 'true', 'output-sort': 'trans', 'obs-time': '2023-12-03 10:41:15', 'maxoutput': '100', 'mag-required': 'true', 'optical': 'true', 'elev-min': '60', 'vmag-max': '13', 'vmag-min': '6'}, 'sb_constraints': {'sb-kind': 'a'}, 'location': 'GSHS Observatory, Suwon', 'obs_night': {'sun_set_az': '242.8', 'sun_set': '2023-12-02 08:13', 'end_civil': '2023-12-02 08:41', 'end_nautical': '2023-12-02 09:14', 'end_astronomical': '2023-12-02 09:45', 'begin_astronomical': '2023-12-02 20:54', 'begin_nautical': '2023-12-02 21:25', 'begin_civil': '2023-12-02 21:58', 'sun_rise_az': '117.3', 'sun_rise': '2023-12-02 22:26', 'transit_phase': '0.72', 'transit': '2023-12-02 19:56 ', 'moon_set_phase': '0.70', 'moon_set': '2023-12-03 03:15 ', 'moon_rise_phase': '0.74', 'moon_rise': '2023-12-02 12:30 ', 'begin_dark': '2023-12-02 09:45', 'mid_dark': '2023-12-02 15

안에 들어 있는 데이터를 확인해 보곘습니다.

In [29]:
print("type(data) :", type(data))
print("data.keys :", data.keys)

type(data) : <class 'dict'>
data.keys : <built-in method keys of dict object at 0x7efd6cce2a00>


 아마도 data['fields']에 있는 것이 필요한 자료인것 같습니다.

In [30]:
print(data['obs_constraints'])
print(data['sb_constraints'])
print(data['fields'])
print(data['data'])
print(type(data['fields']))
print(len(data['fields']))
print(len(data['data'][0]))


{'time-min': '15', 'mpc-code': 'P64', 'fmt-ra-dec': 'true', 'output-sort': 'trans', 'obs-time': '2023-12-03 10:41:15', 'maxoutput': '100', 'mag-required': 'true', 'optical': 'true', 'elev-min': '60', 'vmag-max': '13', 'vmag-min': '6'}
{'sb-kind': 'a'}
['Designation', 'Full name', 'Rise time', 'Transit time', 'Set time', 'Max. time observable', 'R.A.', 'Dec.', 'Vmag', 'Helio. range (au)', 'Topo.range (au)', 'Object-Observer-Sun (deg)', 'Object-Observer-Moon (deg)', 'Galactic latitude (deg)']
[['602', '602 Marianna (A906 DJ)', '08:56*', '10:18', '11:40', '01:55', '23:30:48', '+13 32\'46"', '12.9', '2.31', '1.81', '107.94', '129.8', '-44.86'], ['678', '678 Fredegundis (A909 BQ)', '10:02', '10:53', '11:44', '01:42', '00:05:34', '+09 39\'20"', '12.8', '2.01', '1.39', '114.68', '125.7', '-51.61'], ['639', '639 Latona (A907 OA)', '10:08', '11:54', '13:40', '03:32', '01:06:33', '+18 22\'35"', '13.0', '2.77', '2.02', '131.36', '108.6', '-44.50'], ['123', '123 Brunhild (A872 OB)', '10:17', '12:0

### dataframe 만들기

아무래도 가장 다루기 쉬운 데이타 형태인 dataframe으로 만들어 보겠습니다.

In [31]:
import pandas as pd

df = pd.DataFrame.from_dict(data['data'])
df = df.set_axis((data['fields']), axis=1)
df['OBSdate(UT)'] = obs_date
df

,Designation,Full name,Rise time,Transit time,Set time,Max. time observable,R.A.,Dec.,Vmag,Helio. range (au),Topo.range (au),Object-Observer-Sun (deg),Object-Observer-Moon (deg),Galactic latitude (deg),OBSdate(UT)
0,602,602 Marianna (A906 DJ),08:56*,10:18,11:40,01:55,23:30:48,"+13 32'46""",12.9,2.31,1.81,107.94,129.8,-44.86,20231203
1,678,678 Fredegundis (A909 BQ),10:02,10:53,11:44,01:42,00:05:34,"+09 39'20""",12.8,2.01,1.39,114.68,125.7,-51.61,20231203
2,639,639 Latona (A907 OA),10:08,11:54,13:40,03:32,01:06:33,"+18 22'35""",13.0,2.77,2.02,131.36,108.6,-44.50,20231203
3,123,123 Brunhild (A872 OB),10:17,12:02,13:48,03:30,01:15:04,"+18 13'43""",12.9,2.41,1.62,133.29,106.9,-44.45,20231203
4,135,135 Hertha (A874 DA),10:58,12:09,13:19,02:21,01:21:31,"+11 51'59""",11.9,2.23,1.43,133.01,108.6,-50.51,20231203
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
58,63,63 Ausonia (A861 CA),19:49,21:09*,22:29*,01:04,10:23:47,"+13 14'07""",12.0,2.62,2.31,97.28,20.65,52.63,20231203
59,372,372 Palma (A893 QB),19:33,21:19*,23:05*,01:20,10:33:41,"+18 20'48""",12.2,2.52,2.20,96.88,21.18,56.92,20231203
60,349,349 Dembowska (A892 XB),20:24,21:52*,23:19*,00:30,11:06:05,"+14 33'00""",11.5,3.06,2.93,88.25,29.68,62.25,20231203
61,31,31 Euphrosyne (A854 RA),19:26,21:58*,00:30*,01:28,11:12:17,"+38 02'22""",11.8,2.55,2.25,96.31,31.97,66.25,20231203


### 관측 계획 저장 폴더 생성

FITS 파일을 저장할 폴더를 "OBSplan_asteroid" 이라는 이름으로 생성해보겠습니다.

* 만약 리눅스 시스템 이라면 shell 명령어로 가능한데, "!"를 붙이면 shell 명령어를 실행할 수 있습니다.
> !mkdir OBSplan_asteroid

OS의 영향을 받지 않기 위하여 pathlib을 사용하여 다음과 같이 폴더를 생성합니다.

In [32]:
import os
from pathlib import Path
BASEPATH = Path("./")
save_dir_name = "OBSplan_asteroid"
print(f"BASEPATH: {BASEPATH}")

if not (BASEPATH/save_dir_name).exists():
    os.mkdir(str(BASEPATH/save_dir_name))
    print (f"{str(BASEPATH/save_dir_name)} is created...")
else :
    print (f"{str(BASEPATH/save_dir_name)} is already exist...")

BASEPATH: .
OBSplan_asteroid is already exist...


### dataframe 저장

In [ ]:
df.to_csv(str(BASEPATH/save_dir_name/ f'asteroid_{obs_date}.csv'))

### (과제)

12월 1달 동안의 NEA 목록을 csv  파일로 저장하는 코드를 완성하세요.



## 소행성 좌표 얻기

### 소행성 id 잘라내기

앞서 얻은 dataframe로 부터 소행성의 이름을 알아보자.

In [ ]:
#print("df :", df)
#print("df['Full name'] :", df['Full name'])
print("df['Full name'][0] :", df['Full name'][0])

#import re
#asteroid_names = re.findall('\(([^)]+)', df['Full name'][0])   #extracts string in bracket()
#print (asteroid_names[0])

asteroid_id = df['Full name'][0].split(" ")[0]   #extracts string in bracket()
print("asteroid_id :", asteroid_id)
obs_date = df['OBSdate(UT)'][0]

df['Full name'][0] : 602 Marianna (A906 DJ)
asteroid_id : 602


### astroquery.mpc

[astroquery.mpc](https://astroquery.readthedocs.io/en/latest/mpc/mpc.html) 를 이용하여 시간에 따른 소행성의 좌표를 얻어보자.

In [ ]:
from astroquery.mpc import MPC
eph = MPC.get_ephemeris(asteroid_id, step='1h', number=30 * 24)
print("type(eph) :", type(eph))
print("eph :", eph)
df_eph = eph.to_pandas()
df_eph

type(eph) : <class 'astropy.table.table.Table'>
eph :           Date                  RA         ... Uncertainty 3sig Unc. P.A.
                               deg         ...      arcsec         deg   
----------------------- ------------------ ... ---------------- ---------
2023-12-01 09:00:00.000  352.1704166666666 ...                0      49.2
2023-12-01 10:00:00.000 352.17791666666665 ...                0      49.2
2023-12-01 11:00:00.000  352.1854166666666 ...                0      49.2
2023-12-01 12:00:00.000 352.19333333333327 ...                0      49.2
2023-12-01 13:00:00.000 352.20083333333326 ...                0      49.2
2023-12-01 14:00:00.000 352.20833333333326 ...                0      49.2
2023-12-01 15:00:00.000 352.21583333333325 ...                0      49.2
2023-12-01 16:00:00.000  352.2233333333333 ...                0      49.2
2023-12-01 17:00:00.000 352.23124999999993 ...                0      49.2
2023-12-01 18:00:00.000  352.2387499999999 ...            

,Date,RA,Dec,Delta,r,Elongation,Phase,V,Proper motion,Direction,Uncertainty 3sig,Unc. P.A.
0,2023-12-01 09:00:00,352.170417,13.355000,1.793,2.310,108.9,23.8,12.9,27.22,75.5,0,49.2
1,2023-12-01 10:00:00,352.177917,13.356944,1.793,2.310,108.9,23.8,12.9,27.26,75.5,0,49.2
2,2023-12-01 11:00:00,352.185417,13.358889,1.794,2.310,108.9,23.8,12.9,27.30,75.5,0,49.2
3,2023-12-01 12:00:00,352.193333,13.360556,1.794,2.310,108.8,23.8,12.9,27.33,75.5,0,49.2
4,2023-12-01 13:00:00,352.200833,13.362500,1.795,2.310,108.8,23.8,12.9,27.37,75.5,0,49.2
...,...,...,...,...,...,...,...,...,...,...,...,...
715,2023-12-31 04:00:00,359.852083,15.559444,2.151,2.324,87.4,25.0,13.4,49.05,72.2,0,51.4
716,2023-12-31 05:00:00,359.865417,15.563889,2.151,2.324,87.4,25.0,13.4,49.07,72.2,0,51.4
717,2023-12-31 06:00:00,359.878750,15.568056,2.152,2.324,87.4,25.0,13.4,49.10,72.2,0,51.4
718,2023-12-31 07:00:00,359.892500,15.572222,2.152,2.324,87.3,25.0,13.4,49.12,72.2,0,51.4


### 최대 움직임 속도 알아보기

In [ ]:
print(f"Proper motion: {eph['Proper motion'].quantity.to('arcmin/h').max():.03f}")

Proper motion: 0.819 arcmin / h


### 파일 저장하기

In [ ]:
eph = MPC.get_ephemeris(asteroid_id, location='P64',
                        ra_format={'sep': ':', 'unit': 'hourangle', 'precision': 1},
                        dec_format={'sep': ':', 'precision': 0},
                        step='1h')
print("type(eph): ", type(eph))
#print("eph: ", eph)
df_eph = eph.to_pandas()
df_eph['Asteroid ID'] = asteroid_id
df_eph['Asteroid name'] = df['Full name'][0]
df_eph.to_csv(str(BASEPATH/save_dir_name/ f'asteroid_{asteroid_id}.csv'))
df_eph

type(eph):  <class 'astropy.table.table.Table'>


,Date,RA,Dec,Delta,r,Elongation,Phase,V,Proper motion,Direction,Azimuth,Altitude,Sun altitude,Moon phase,Moon distance,Moon altitude,Uncertainty 3sig,Unc. P.A.,Asteroid ID,Asteroid name
0,2023-12-01 09:00:00,23:28:41.0,13:21:16,1.793,2.31,108.9,23.8,12.9,26.31,74.8,138,60,-9,0.84,115,-20,0,49.2,602,602 Marianna (A906 DJ)
1,2023-12-01 10:00:00,23:28:42.8,13:21:23,1.793,2.31,108.9,23.8,12.9,26.28,74.9,168,66,-21,0.84,116,-13,0,49.2,602,602 Marianna (A906 DJ)
2,2023-12-01 11:00:00,23:28:44.5,13:21:30,1.794,2.31,108.9,23.8,12.9,26.31,75.0,203,65,-33,0.83,116,-4,0,49.2,602,602 Marianna (A906 DJ)
3,2023-12-01 12:00:00,23:28:46.2,13:21:36,1.794,2.31,108.8,23.8,12.9,26.41,75.2,230,57,-45,0.83,117,6,0,49.2,602,602 Marianna (A906 DJ)
4,2023-12-01 13:00:00,23:28:48.0,13:21:43,1.795,2.31,108.8,23.8,12.9,26.57,75.4,247,47,-56,0.83,117,16,0,49.2,602,602 Marianna (A906 DJ)
5,2023-12-01 14:00:00,23:28:49.8,13:21:50,1.795,2.31,108.8,23.8,12.9,26.79,75.6,259,36,-67,0.82,118,27,0,49.2,602,602 Marianna (A906 DJ)
6,2023-12-01 15:00:00,23:28:51.6,13:21:56,1.796,2.31,108.7,23.8,12.9,27.05,75.8,269,24,-74,0.82,118,38,0,49.2,602,602 Marianna (A906 DJ)
7,2023-12-01 16:00:00,23:28:53.4,13:22:03,1.796,2.31,108.7,23.9,12.9,27.34,75.9,278,12,-72,0.82,118,50,0,49.2,602,602 Marianna (A906 DJ)
8,2023-12-01 17:00:00,23:28:55.2,13:22:10,1.797,2.31,108.7,23.9,12.9,27.64,76.0,287,0,-63,0.81,119,61,0,49.2,602,602 Marianna (A906 DJ)
9,2023-12-01 18:00:00,23:28:57.0,13:22:16,1.797,2.31,108.6,23.9,12.9,27.93,76.1,296,-11,-52,0.81,119,71,0,49.2,602,602 Marianna (A906 DJ)


### (과제)

df에 있는 소행성 목록들의 eph를 csv 파일로 저장해 보자.

In [ ]:
from pathlib import Path
from astroquery.mpc import MPC
from pprint import pprint
import re

for i, row  in df.iterrows():
    print("row['Full name'] :", row["Full name"])

    #asteroid_names = re.findall('\(([^)]+)', df['Full name'][0])   #extracts string in bracket()
    #print (asteroid_names[0])
    asteroid_id = df['Full name'][i].split(" ")[0]   #extracts string
    print(asteroid_id)

    eph = MPC.get_ephemeris(asteroid_id, step='1h', number=30 * 24)
    print(f"Proper motion: {eph['Proper motion'].quantity.to('arcmin/h').max():.03f}")
    eph = MPC.get_ephemeris(asteroid_id, location='P64',
                        ra_format={'sep': ':', 'unit': 'hourangle', 'precision': 1},
                        dec_format={'sep': ':', 'precision': 0},
                        step='1h')
    print("type(eph): ", type(eph))
    #print("eph: ", eph)

    df_eph = eph.to_pandas()
    df_eph['Asteroid ID'] = asteroid_id
    df_eph['Asteroid name'] = df['Full name'][i]
    df_eph.to_csv(str(BASEPATH/save_dir_name/ f"eph_asteroid_{asteroid_id}.csv"))

row['Full name'] : 602 Marianna (A906 DJ)
602
Proper motion: 0.819 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 678 Fredegundis (A909 BQ)
678
Proper motion: 0.911 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 639 Latona (A907 OA)
639
Proper motion: 0.390 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 123 Brunhild (A872 OB)
123
Proper motion: 0.437 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 135 Hertha (A874 DA)
135
Proper motion: 0.500 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 122 Gerda (A872 OA)
122
Proper motion: 0.227 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 127 Johanna (A872 VB)
127
Proper motion: 0.294 arcmin / h
type(eph):  <class 'astropy.table.table.Table'>
row['Full name'] : 509 Iolanda (A903 HD)
509
Proper motion: 0.308 arcmin / h
type(eph):  <class 'astropy.table.table.Table'

저장

In [ ]:
eph = MPC.get_ephemeris('A894 VC', start='1996-03-01', step='1h', number=30 * 24)
print(eph['Proper motion'].quantity.to('deg/h').max())
eph = MPC.get_ephemeris('A894 VC', location='P64',
                        ra_format={'sep': ':', 'unit': 'hourangle', 'precision': 1},
                        dec_format={'sep': ':', 'precision': 0},
                        step='1h')
print(eph)

0.023877777777777776 deg / h
          Date              RA       Dec    ... Uncertainty 3sig Unc. P.A.
                        hourangle    deg    ...      arcsec         deg   
----------------------- ---------- -------- ... ---------------- ---------
2023-12-01 09:00:00.000 21:23:21.4 -4:53:29 ...                0      77.0
2023-12-01 10:00:00.000 21:23:26.6 -4:53:22 ...                0      77.0
2023-12-01 11:00:00.000 21:23:31.7 -4:53:15 ...                0      77.0
2023-12-01 12:00:00.000 21:23:36.9 -4:53:08 ...                0      77.0
2023-12-01 13:00:00.000 21:23:42.1 -4:53:01 ...                0      77.0
2023-12-01 14:00:00.000 21:23:47.3 -4:52:54 ...                0      77.0
2023-12-01 15:00:00.000 21:23:52.5 -4:52:47 ...                0      77.1
2023-12-01 16:00:00.000 21:23:57.7 -4:52:40 ...                0      77.0
2023-12-01 17:00:00.000 21:24:03.0 -4:52:32 ...                0      77.1
2023-12-01 18:00:00.000 21:24:08.2 -4:52:25 ...                0      7